In [1]:
from src.data.dataset import BCCDDataset

import polars as pl
import numpy as np

In [2]:
BASE = "../data/raw"
TRAIN = "train"
TEST = "test"
VAL = "val"
IMG = "img"
ANN = "ann"
META = "meta.json"
EXT = (".json", ".jpeg")

train_df = pl.read_csv("../data/processed/train_data.csv")

In [3]:
train_df.head()

file_id,object_id,img_height,img_width,xmin,ymin,xmax,ymax,label,label_name,image_file,ann_file,bb_height,bb_width,bb_area,bb_aspectr
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,i64,i64,i64,f64
"""BloodImage_00001""",7789770,480,640,68,315,286,479,2,"""WBC""","""BloodImage_00001.jpeg""","""BloodImage_00001.jpeg.json""",164,218,35752,1.329268
"""BloodImage_00001""",7789769,480,640,346,361,446,454,1,"""RBC""","""BloodImage_00001.jpeg""","""BloodImage_00001.jpeg.json""",93,100,9300,1.075269
"""BloodImage_00001""",7789768,480,640,53,179,146,299,1,"""RBC""","""BloodImage_00001.jpeg""","""BloodImage_00001.jpeg.json""",120,93,11160,0.775
"""BloodImage_00001""",7789767,480,640,449,400,536,479,1,"""RBC""","""BloodImage_00001.jpeg""","""BloodImage_00001.jpeg.json""",79,87,6873,1.101266
"""BloodImage_00001""",7789766,480,640,461,132,548,212,1,"""RBC""","""BloodImage_00001.jpeg""","""BloodImage_00001.jpeg.json""",80,87,6960,1.0875


In [4]:
import os

In [5]:
dataset = BCCDDataset(train_df, image_base=os.path.join(BASE, TRAIN, IMG))

In [6]:
dataset[1][0].shape

(480, 640, 3)

In [7]:
from src.data.transforms import get_train_transforms

In [8]:
dataset2 = BCCDDataset(train_df, image_base=os.path.join(BASE, TRAIN, IMG), transforms=get_train_transforms(512))

In [13]:
dataset2[6][0].shape

torch.Size([3, 512, 512])

In [12]:
dataset2[6][1]

{'boxes': tensor([[136.0000, 218.4000, 224.0000, 300.0000],
         [321.6000, 194.4000, 409.6000, 276.0000],
         [400.8000, 124.0000, 488.8000, 205.6000],
         [393.6000, 316.0000, 481.6000, 397.6000],
         [  0.8000, 339.2000,  88.8000, 423.2000],
         [118.4000, 128.0000, 216.0000, 210.4000],
         [210.4000, 303.2000, 308.0000, 385.6000],
         [340.8000, 261.6000, 423.2000, 330.4000],
         [112.0000, 379.2000, 194.4000, 447.2000],
         [ 52.8000, 297.6000,  88.8000, 332.8000],
         [208.0000,  64.8000, 387.2000, 214.4000]]),
 'labels': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 2]),
 'area': tensor([ 7180.8008,  7180.8008,  7180.7993,  7180.8008,  7392.0000,  8042.2407,
          8042.2397,  5669.1206,  5603.1997,  1267.1995, 26808.3184]),
 'image_id': tensor([6]),
 'iscrowd': tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])}

In [14]:
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights

In [15]:
backbone = mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.IMAGENET1K_V1)

Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-8738ca79.pth" to C:\Users\apmal/.cache\torch\hub\checkpoints\mobilenet_v3_large-8738ca79.pth


100.0%


In [16]:
backbone

MobileNetV3(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): Hardswish()
    )
    (1): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (1): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
        )
      )
    )
    (2): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 64, kernel_size=(1, 1), stride=(1, 1), bi

In [17]:
from torchvision.models import mobilenet_v2

In [18]:
base_model = mobilenet_v2(weights=None)
base_model

MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=

In [1]:
from src.models.vision_finetuned import get_model_objectdetection_mobilenet

In [2]:
model = get_model_objectdetection_mobilenet(device="cuda")

Total parameters: 17,549,698
Trainable parameters: 17,549,698


In [3]:
model

FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
        (2): Hardswish()
      )
      (1): InvertedResidual(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
            (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
            (2): ReLU(inplace=True)
          )
          (1): Conv2dNormActivation(
            (0): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (1): BatchNorm2d(